## Lakelink Fiber — Margin Truth SDP Pipeline

**Target:** `cdm_tmforum.revenue_assurance` (no new schemas)  
**Layers:** Bronze (Auto Loader → PDFs) → Silver (ai_parse_document + ai_extract) → Gold (margin_ledger with expectations)  
**Tag:** `telco_project = margin_analysis`  
**Task:** P1-SDP-PIPELINE

In [0]:
%sql
-- Bronze: Stream carrier invoice PDFs from the margin_documents volume via Auto Loader.
-- Grain: one row per PDF file. Format: binaryFile (path, content, length, modificationTime).
CREATE OR REFRESH STREAMING TABLE margin_bronze_invoice_docs
COMMENT 'Raw carrier invoice PDFs ingested via Auto Loader from the margin_documents UC volume. Grain: one row per PDF file. Source: /Volumes/cdm_tmforum/revenue_assurance/margin_documents/carrier_invoices/'
TBLPROPERTIES ('telco_project' = 'margin_analysis')
AS SELECT
  path,
  content,
  length,
  modificationTime
FROM STREAM(read_files(
  '/Volumes/cdm_tmforum/revenue_assurance/margin_documents/carrier_invoices/',
  format => 'binaryFile'
));

In [0]:
%sql
-- Silver: Apply ai_parse_document then ai_extract to pull structured carrier invoice fields.
-- Grain: one row per PDF. Extracted fields: invoice_number, vendor_name, circuit_id, invoice_month,
-- invoiced_rate_usd, vendor_contract_rate_usd, paid_amount_usd.
CREATE OR REFRESH MATERIALIZED VIEW margin_silver_invoice_parsed
COMMENT 'Parsed carrier invoice PDFs with structured fields extracted via ai_parse_document and ai_extract. Grain: one row per PDF. Enables scene 2 lineage trace from ledger back to source document.'
TBLPROPERTIES ('telco_project' = 'margin_analysis')
AS
WITH parsed AS (
  SELECT
    path,
    ai_parse_document(content) AS parsed_doc
  FROM margin_bronze_invoice_docs
)
SELECT
  path,
  ai_extract(
    parsed_doc,
    '{
      "invoice_number": {"type": "string", "description": "Carrier invoice number (LLF-CI-... format)"},
      "vendor_name":    {"type": "string", "description": "Third-party carrier or vendor name"},
      "circuit_id":     {"type": "string", "description": "Circuit identifier (CKT-XXXXX format)"},
      "invoice_month":  {"type": "string", "description": "Invoice period in YYYY-MM format"},
      "invoiced_rate_usd":        {"type": "number", "description": "Rate invoiced by the carrier in USD"},
      "vendor_contract_rate_usd": {"type": "number", "description": "Contracted rate with the carrier in USD"},
      "paid_amount_usd":          {"type": "number", "description": "Amount paid to the carrier in USD"}
    }',
    MAP('version', '2.1', 'instructions', 'Extract carrier invoice fields from a Lakelink Fiber carrier invoice PDF. Each PDF contains one invoice line for one circuit.')
  ) AS extracted
FROM parsed;

In [0]:
%sql
-- Gold: The margin ledger. Same schema and logic as the fast-path SQL (DEC-008),
-- now pipeline-managed. Joins margin_billing, margin_carrier_invoices, margin_contracts,
-- margin_circuits, margin_products, margin_customers, margin_gl_actuals, margin_allocation_rules.
-- Expectations:
--   pk_not_null: composite PK columns must be non-null (FAIL on violation)
--   no_carrier_overcharge: carrier_rate_variance_usd <= 0 (WARN: planted story, ~4500 rows expected to violate)
--   positive_revenue: revenue_usd > 0 (DROP null/zero revenue rows)
CREATE OR REFRESH MATERIALIZED VIEW margin_ledger (
  CONSTRAINT pk_not_null EXPECT (ledger_month IS NOT NULL AND customer_id IS NOT NULL AND product_id IS NOT NULL AND circuit_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT no_carrier_overcharge EXPECT (carrier_rate_variance_usd <= 0),
  CONSTRAINT positive_revenue EXPECT (revenue_usd > 0) ON VIOLATION DROP ROW
)
COMMENT 'Lakelink Fiber margin ledger (gold): revenue, network expense, cost to serve and allocation drivers per circuit per month. Grain: customer x product x circuit x month. Source: margin_billing, margin_carrier_invoices, margin_contracts, margin_circuits, margin_products, margin_customers, margin_gl_actuals, margin_allocation_rules. Contribution margin is defined in cdm_tmforum._metrics.margin_ledger_mv (DEC-002). Replaces the fast-path SQL table (DEC-008); same name and columns.'
TBLPROPERTIES ('telco_project' = 'margin_analysis')
AS
WITH base AS (
  SELECT
    b.billing_month                          AS ledger_month,
    b.customer_id,
    k.product_id,
    b.circuit_id,
    b.contract_id,
    c.vendor_id,
    c.is_on_net,
    p.product_family,
    p.family_class,
    u.segment,
    u.region,
    CAST(b.billed_amount_usd   AS DECIMAL(18,2)) AS revenue_usd,
    CAST(b.expected_amount_usd AS DECIMAL(18,2)) AS expected_revenue_usd,
    CAST(
      CASE
        WHEN c.is_on_net
          THEN k.contract_rate_usd * CASE WHEN p.family_class = 'legacy' THEN 0.60 ELSE 0.25 END
        ELSE COALESCE(ci.paid_amount_usd, 0)
      END AS DECIMAL(18,2)
    ) AS network_expense_usd,
    CAST(
      CASE
        WHEN c.is_on_net OR ci.circuit_id IS NULL THEN 0
        ELSE ci.invoiced_rate_usd - ci.vendor_contract_rate_usd
      END AS DECIMAL(18,2)
    ) AS carrier_rate_variance_usd,
    1 AS circuit_count,
    CAST(c.bandwidth_mbps AS DECIMAL(12,2)) AS bandwidth_mbps,
    CAST(
      CASE
        WHEN p.family_class = 'legacy'
          THEN FLOOR(PMOD(XXHASH64(b.circuit_id, b.billing_month), 1000) / 250)
        ELSE
          CASE WHEN PMOD(XXHASH64(b.circuit_id, b.billing_month), 100) < 25 THEN 1 ELSE 0 END
      END AS INT
    ) AS ticket_count
  FROM      cdm_tmforum.revenue_assurance.margin_billing          b
  JOIN      cdm_tmforum.revenue_assurance.margin_circuits          c  ON c.circuit_id  = b.circuit_id
  JOIN      cdm_tmforum.revenue_assurance.margin_contracts         k  ON k.contract_id = b.contract_id
  JOIN      cdm_tmforum.revenue_assurance.margin_products          p  ON p.product_id  = k.product_id
  JOIN      cdm_tmforum.revenue_assurance.margin_customers         u  ON u.customer_id = b.customer_id
  LEFT JOIN cdm_tmforum.revenue_assurance.margin_carrier_invoices ci  ON ci.circuit_id = b.circuit_id
                                                                    AND ci.invoice_month = b.billing_month
),
tot AS (
  SELECT ledger_month,
         SUM(circuit_count)   AS s_cc,
         SUM(bandwidth_mbps)  AS s_bw,
         SUM(ticket_count)    AS s_tk,
         SUM(revenue_usd)     AS s_rev
  FROM base
  GROUP BY ALL
),
pools AS (
  SELECT g.period_month AS ledger_month,
         r.driver,
         SUM(g.amount_usd * r.weight) AS amt
  FROM cdm_tmforum.revenue_assurance.margin_gl_actuals        g
  JOIN cdm_tmforum.revenue_assurance.margin_allocation_rules  r
    ON r.cost_pool = g.cost_pool
   AND r.is_active
   AND g.period_month >= r.effective_from
   AND (r.effective_to IS NULL OR g.period_month <= r.effective_to)
  WHERE g.account_class = 'cost_to_serve_pool'
  GROUP BY ALL
),
pw AS (
  SELECT ledger_month,
         SUM(CASE WHEN driver = 'circuit_count'  THEN amt END) AS a_cc,
         SUM(CASE WHEN driver = 'bandwidth_mbps' THEN amt END) AS a_bw,
         SUM(CASE WHEN driver = 'ticket_count'   THEN amt END) AS a_tk,
         SUM(CASE WHEN driver = 'billed_revenue' THEN amt END) AS a_rev
  FROM pools
  GROUP BY ALL
)
SELECT
  b.ledger_month, b.customer_id, b.product_id, b.circuit_id, b.contract_id,
  b.vendor_id, b.is_on_net, b.product_family, b.family_class, b.segment, b.region,
  b.revenue_usd,
  b.network_expense_usd,
  CAST(
    COALESCE(pw.a_cc, 0) * b.circuit_count  / t.s_cc
  + COALESCE(pw.a_bw, 0) * b.bandwidth_mbps / t.s_bw
  + COALESCE(pw.a_tk, 0) * b.ticket_count   / NULLIF(t.s_tk, 0)
  + COALESCE(pw.a_rev,0) * b.revenue_usd    / t.s_rev
  AS DECIMAL(18,2)) AS cost_to_serve_usd,
  b.expected_revenue_usd,
  b.carrier_rate_variance_usd,
  b.circuit_count,
  b.bandwidth_mbps,
  b.ticket_count,
  b.revenue_usd AS billed_revenue_usd,
  'MGN-ALR-v1'          AS allocation_rule_version,
  'MIXED'               AS origin,
  'margin_sdp_pipeline' AS source_system
FROM base b
JOIN tot  t  USING (ledger_month)
LEFT JOIN pw USING (ledger_month);